## 将文本转换为词元

`DistilBERT`这种`Transformer`模型是不能接收原始字符串作为输入的，需要文本已经词元化(`Tokenization`)并编码为为数字向量。

`Tokenizaiton`是将字符串分解为模型使用的原子单元的步骤。   
直接字符词元化和单词词元化，是2种极端的情况。也是便于理解的`Tokenization`。

In [1]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.set_printoptions(edgeitems=2)
torch.manual_seed(123456)

print("")

### 1. 字符词元化
> 直接把每一个字符用独热编码表示。

#### 1.1 得到字符索引字典

In [2]:
text = "PyTorch is an optimized tensor library for deep learning using GPUs and CPUs."

# 直接把str转化为一个list
text_tokenized = list(text)

# 次元的唯一集合
text_tokenized_set = set(text_tokenized)

In [3]:
len(text_tokenized), len(text_tokenized_set)

(77, 27)

In [4]:
# 我们把每个字符给一个数字
token_to_index_dict = {char: index for index, char in enumerate(sorted(text_tokenized_set))}
token_to_index_dict

{' ': 0,
 '.': 1,
 'C': 2,
 'G': 3,
 'P': 4,
 'T': 5,
 'U': 6,
 'a': 7,
 'b': 8,
 'c': 9,
 'd': 10,
 'e': 11,
 'f': 12,
 'g': 13,
 'h': 14,
 'i': 15,
 'l': 16,
 'm': 17,
 'n': 18,
 'o': 19,
 'p': 20,
 'r': 21,
 's': 22,
 't': 23,
 'u': 24,
 'y': 25,
 'z': 26}

**根据不同的数据集/语料库，得到的字符和索引字典是不一样的**

#### 1.2 根据字符生成都热编码

In [5]:
# 根据字符、字符索引字典，将字符转换为一个数值列表
input_ids = [token_to_index_dict.get(token, 0) for token in text_tokenized]
input_ids[:5], text_tokenized[:5]

([4, 25, 5, 19, 21], ['P', 'y', 'T', 'o', 'r'])

**将text_ids转换为独热编码**

##### 方式一：

In [6]:
one_hot_encodings = torch.zeros(len(input_ids), len(text_tokenized_set))
one_hot_encodings.shape

torch.Size([77, 27])

In [7]:
# 修改其中每个值
for i, token in enumerate(input_ids):
    # 得到：序号和，token对应的数字值
    one_hot_encodings[i][token] = 1

one_hot_encodings[0], input_ids[0]

(tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 4)

#### 方式2：使用torch中的函数

```python
import torch.nn.functional as F
```

In [8]:
# 先生成字符列表张量
text_ids_tensor = torch.tensor(input_ids)

# 直接调用函数
one_hot_encodings_2 = F.one_hot(text_ids_tensor, num_classes=len(text_tokenized_set))

# 查看形状
one_hot_encodings_2.shape

torch.Size([77, 27])

**到这里**，我们就有了77个输入次元。  
得到一个27维的独热向量，27是唯一字符的数量。

In [9]:
# 查看第1个元素
one_hot_encodings_2[0], text_ids_tensor[0]

(tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]),
 tensor(4))

### 2. 单词词元化

和上面的字符次元相比，单词词元化是将文本细分为单词，并将每个单词映射到一个整数。    

In [10]:
text

'PyTorch is an optimized tensor library for deep learning using GPUs and CPUs.'

In [11]:
tokenized_word = text.split()
print(tokenized_word)

['PyTorch', 'is', 'an', 'optimized', 'tensor', 'library', 'for', 'deep', 'learning', 'using', 'GPUs', 'and', 'CPUs.']


In [12]:
# 把单词映射到整数
tokenized_word_set = set(tokenized_word)

# 映射字典
word_to_ids_dict = {word: idx for idx, word in enumerate(sorted(tokenized_word_set))}

# 单词转换为整数
word_input_ids = [word_to_ids_dict[word] for word in tokenized_word]

In [13]:
print(word_input_ids)

[2, 7, 3, 10, 11, 9, 6, 5, 8, 12, 1, 4, 0]


In [14]:
# 单词独热编码
word_input_ids = torch.tensor(word_input_ids)
word_one_hot_encodings = F.one_hot(word_input_ids, num_classes=len(tokenized_word_set))
word_one_hot_encodings.shape

torch.Size([13, 13])

字符组合的单词，是有大量的组合的。这样就会是一个问题了。把这些作为输入传递给神经网络，会有大量的参数。   
所以，单词词元化，只是为了理解`Tokenization`，真实项目中是不会这样用的。

单个字符词元化是参数太少，单词词元化是参数太多了。    
那么，接下来就引出：**子词词元化**。   

### 3. 子词词元化
子词词元化其基本思想是：将字符词元化和单词词元化的优点结合起来。   
子词词元化是使用统计规则和算法从预训练的语料库中学习得到的。

In [15]:
# 直接使用模型体验子词词元化
from transformers import AutoTokenizer

In [16]:
model_name = "distilbert-base-uncased"

# 第一次执行会下载模型
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [17]:
encoded_text = tokenizer(text)
print(encoded_text)

{'input_ids': [101, 1052, 22123, 2953, 2818, 2003, 2019, 23569, 27605, 5422, 23435, 3075, 2005, 2784, 4083, 2478, 14246, 2271, 1998, 17368, 2015, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [18]:
# 把数字转换为Token字符
tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(tokens)

['[CLS]', 'p', '##yt', '##or', '##ch', 'is', 'an', 'opt', '##imi', '##zed', 'tensor', 'library', 'for', 'deep', 'learning', 'using', 'gp', '##us', 'and', 'cpu', '##s', '.', '[SEP]']


从上面的输出我们看到了一些特殊的次元：`[CLS]`、`[SEP]`。    
这些特殊的次元根据模型不同而不同的，这里的`[CLS]`和`[SEP]`是指示序列的开始和结束。

In [19]:
# 把数字转换为字符
tokens_string = tokenizer.convert_tokens_to_string(tokens)
print(tokens_string)

[CLS] pytorch is an optimized tensor library for deep learning using gpus and cpus. [SEP]


`tokenizer`的一些其他属性。

In [20]:
# 查看检查此表的大小
tokenizer.vocab_size

30522

In [21]:
# 模型最大上下文大小
tokenizer.model_max_length

512

In [22]:
# 模型在前向传播中期望的字段名字：
tokenizer.model_input_names

['input_ids', 'attention_mask']